# Task 3: A/B Hypothesis Testing

AlphaCare Insurance Solutions (ACIS)

Statistical validation of risk drivers across provinces, zip codes, and gender.

Null hypotheses to test:
1. H₀: No risk differences across provinces (claim frequency).
2. H₀: No risk differences between zip codes (claim frequency).
3. H₀: No significant margin (profit) difference between zip codes.
4. H₀: No significant risk difference between women and men (claim frequency).

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu

# Import custom test functions from src/hypothesis_tests.py
from src.hypothesis_tests import chi2_test_frequency, t_test_severity, mannwhitney_margin

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the cleaned data (from Task 2)
df = pd.read_csv('../data/insurance_data_clean.csv')

# Create derived columns
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print(f"Dataset shape: {df.shape}")
print(f"Claim frequency (any claim): {df['HasClaim'].mean():.3f}")
print(f"Mean margin: {df['Margin'].mean():.2f}")

## 2. Hypothesis 1: Risk differences across provinces

H₀: No risk differences across provinces (claim frequency).

To isolate the effect, we compare two provinces with similar vehicle/plan profiles. Here we compare Gauteng vs Western Cape.

In [ ]:
# Filter to the two provinces
province_subset = df[df['Province'].isin(['Gauteng', 'Western Cape'])].copy()
print(f"Samples: Gauteng = {sum(province_subset['Province'] == 'Gauteng')}, "
      f"Western Cape = {sum(province_subset['Province'] == 'Western Cape')}")

# Run chi-squared test
result1 = chi2_test_frequency(province_subset, 'Province')
result1

Interpretation:
- If p < 0.05, we reject H₀ → provinces have significantly different claim frequencies.
- Business action: Adjust premiums based on province risk factors.

## 3. Hypothesis 2: Risk differences between zip codes

H₀: No risk differences between zip codes (claim frequency).

We select the two most frequent zip codes in the dataset for a balanced comparison.

In [ ]:
# Find top two zip codes by frequency
top_zips = df['PostalCode'].value_counts().head(2).index.tolist()
print(f"Comparing zip codes: {top_zips[0]} vs {top_zips[1]}")

zip_subset = df[df['PostalCode'].isin(top_zips)].copy()
result2 = chi2_test_frequency(zip_subset, 'PostalCode')
result2

Interpretation:
- Rejecting H₀ implies zip‑code‑level risk heterogeneity.
- Action: Use zip code as a pricing factor or target low‑risk zip codes for reduced premiums.

## 4. Hypothesis 3: Margin difference between zip codes

H₀: No significant margin (profit) difference between zip codes.

We use the Mann‑Whitney U test (non‑parametric, robust to outliers) on the same two zip codes.

In [ ]:
# Margin comparison
result3 = mannwhitney_margin(zip_subset, 'PostalCode')
result3

Interpretation:
- If p < 0.05, profitability differs by zip code.
- Action: Focus marketing spend on higher‑margin zip codes; consider premium adjustments for low‑margin areas.

## 5. Hypothesis 4: Risk difference between women and men

H₀: No significant risk difference between women and men (claim frequency).

In [ ]:
gender_subset = df[df['Gender'].isin(['Female', 'Male'])].copy()
print(f"Samples: Female = {sum(gender_subset['Gender'] == 'Female')}, "
      f"Male = {sum(gender_subset['Gender'] == 'Male')}")

result4 = chi2_test_frequency(gender_subset, 'Gender')
result4

Interpretation:
- If p > 0.05, we fail to reject H₀ → no evidence of gender‑based risk difference.
- Action: Avoid gender‑based pricing (also compliant with many regulatory frameworks).

## 6. Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Hypothesis': 'Province risk differences', 
     'Test': result1['test'], 
     'P-value': result1['p_value'], 
     'Decision': result1['decision']},
    {'Hypothesis': 'Zip code risk differences', 
     'Test': result2['test'], 
     'P-value': result2['p_value'], 
     'Decision': result2['decision']},
    {'Hypothesis': 'Zip code margin differences', 
     'Test': result3['test'], 
     'P-value': result3['p_value'], 
     'Decision': result3['decision']},
    {'Hypothesis': 'Gender risk differences', 
     'Test': result4['test'], 
     'P-value': result4['p_value'], 
     'Decision': result4['decision']}
])

summary

## 7. Business Recommendations

### Based on the results above:

| Hypothesis | Decision | Business Action |
|------------|----------|----------------|
| Province risk differences | (see p‑value) | If rejected: Implement provincial risk multipliers. E.g., Gauteng +10% premium. |
| Zip code risk differences | (see p‑value) | If rejected: Use zip code as a pricing factor; identify low‑risk zip codes for targeted discounts. |
| Zip code margin differences | (see p‑value) | If rejected: Reallocate marketing budget toward high‑margin zip codes. |
| Gender risk differences | (see p‑value) | If not rejected: Maintain gender‑neutral pricing (avoids regulatory risk). |

Next steps:
- Use significant categorical drivers (province, zip code) as features in predictive models (Task 4).
- Design A/B tests in production to validate premium changes before full rollout.

## 8. Additional (Optional) Test – Claim Severity by Gender

Even though claim frequency showed no difference, severity (average claim amount) might differ. Here we test severity (using t‑test) for completeness.

In [ ]:
severity_result = t_test_severity(gender_subset, 'Gender')
severity_result

If p < 0.05: Claim severity differs by gender → consider separate severity models even if frequency is similar.